In [ ]:
"""
EchoVerse - Generative AI Audiobook Creation System
Uses IBM Granite 3.2 2B Instruct model and gTTS for audio generation
Optimized for faster processing with optional tone enhancement
"""

# Install required packages
!pip install -q gradio transformers torch gtts accelerate sentencepiece

import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from gtts import gTTS
import os
from datetime import datetime
import re

# Initialize the IBM Granite model
print("Loading IBM Granite model...")
MODEL_NAME = "ibm-granite/granite-3.2-2b-instruct"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# Supported languages for gTTS
LANGUAGES = {
    "English": "en",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Italian": "it",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Korean": "ko",
    "Chinese (Mandarin)": "zh-CN",
    "Arabic": "ar",
    "Hindi": "hi",
    "Dutch": "nl",
    "Polish": "pl",
    "Turkish": "tr",
    "Swedish": "sv",
    "Indonesian": "id",
    "Thai": "th",
    "Vietnamese": "vi",
    "Greek": "el"
}

# Language names for prompts
LANGUAGE_NAMES = {
    "en": "English",
    "es": "Spanish",
    "fr": "French",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "ja": "Japanese",
    "ko": "Korean",
    "zh-CN": "Chinese",
    "ar": "Arabic",
    "hi": "Hindi",
    "nl": "Dutch",
    "pl": "Polish",
    "tr": "Turkish",
    "sv": "Swedish",
    "id": "Indonesian",
    "th": "Thai",
    "vi": "Vietnamese",
    "el": "Greek"
}

# Sensitive content keywords
SENSITIVE_KEYWORDS = {
    "violence": ["kill", "murder", "blood", "stab", "shoot", "gun", "weapon", "attack", "fight",
                 "punch", "beat", "assault", "torture", "violent", "brutally", "slaughter"],
    "abuse": ["abuse", "abusive", "molest", "rape", "sexual assault", "harassment", "victim",
              "suffering", "pain", "hurt", "harm", "injure"],
    "strong_language": ["fuck", "shit", "damn", "hell", "bitch", "bastard", "ass", "crap",
                       "piss", "dick", "cock", "pussy"]
}

def detect_sensitive_content(text):
    """
    Detect sensitive content in the text using keyword matching
    Returns: (has_sensitive_content, warning_message, categories)
    """
    text_lower = text.lower()
    detected_categories = []

    # Check for keywords
    for category, keywords in SENSITIVE_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text_lower:
                detected_categories.append(category)
                break

    # Remove duplicates
    detected_categories = list(set(detected_categories))

    if detected_categories:
        category_names = {
            "violence": "Violence",
            "abuse": "Abuse/Harmful Content",
            "strong_language": "Strong Language/Profanity"
        }

        warning_msg = "⚠️ **SENSITIVE CONTENT DETECTED**\n\n"
        warning_msg += "This text contains:\n"
        for cat in detected_categories:
            warning_msg += f"• {category_names.get(cat, cat)}\n"
        warning_msg += "\n**Warning**: The generated audiobook may contain mature themes. "
        warning_msg += "Listener discretion is advised."

        return True, warning_msg, detected_categories

    return False, "", []

def get_tone_prompt(tone, text):
    """Generate tone-specific prompts for English text only"""

    if tone == "Neutral":
        return f"""Rewrite the following text in a clear, professional, and neutral tone.
Maintain all key information and meaning while making it suitable for audiobook narration.
Keep the same structure and length:

{text}

Rewritten text:"""

    elif tone == "Suspenseful":
        return f"""Rewrite the following text with a suspenseful and thrilling tone.
Add dramatic tension, vivid imagery, and engaging pacing while preserving the core message.
Make it captivating for audiobook listeners:

{text}

Rewritten text:"""

    elif tone == "Inspiring":
        return f"""Rewrite the following text with an inspiring and motivational tone.
Add uplifting language, powerful imagery, and encouraging elements while keeping the original meaning.
Make it energizing and impactful for audiobook listeners:

{text}

Rewritten text:"""

def generate_rewritten_text(text, tone, temperature=0.7, max_tokens=1024):
    """
    Rewrite text using IBM Granite model with specified tone (English only)
    """
    try:
        # Get the appropriate prompt
        prompt = get_tone_prompt(tone, text)

        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generate rewritten text
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract only the rewritten part (after the prompt)
        if "Rewritten text:" in generated_text:
            rewritten = generated_text.split("Rewritten text:")[-1].strip()
        else:
            # Fallback: take text after the original input
            rewritten = generated_text[len(prompt):].strip()

        return rewritten if rewritten else generated_text

    except Exception as e:
        return f"Error during text generation: {str(e)}"

def text_to_speech(text, lang_code='en', slow=False):
    """
    Convert text to speech using gTTS
    """
    try:
        # Create timestamp for unique filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        audio_file = f"audiobook_{timestamp}.mp3"

        # Generate speech
        tts = gTTS(text=text, lang=lang_code, slow=slow)
        tts.save(audio_file)

        return audio_file
    except Exception as e:
        return f"Error generating audio: {str(e)}"

def process_audiobook(input_text, file_input, tone, speed, language, enable_tone):
    """
    Main processing function for the EchoVerse system
    """
    # Determine text source
    if file_input is not None:
        try:
            with open(file_input.name, 'r', encoding='utf-8') as f:
                original_text = f.read()
        except Exception as e:
            return "Error reading file", f"Error: {str(e)}", None, gr.update(visible=False), gr.update(visible=False)
    elif input_text and input_text.strip():
        original_text = input_text
    else:
        return "No input provided", "Please provide text or upload a file", None, gr.update(visible=False), gr.update(visible=False)

    # Validate text length
    if len(original_text.strip()) < 10:
        return original_text, "Text is too short. Please provide at least 10 characters.", None, gr.update(visible=False), gr.update(visible=False)

    if len(original_text) > 5000:
        return original_text, "Text is too long. Please provide less than 5000 characters.", None, gr.update(visible=False), gr.update(visible=False)

    # Check for sensitive content
    print("🔍 Checking for sensitive content...")
    has_sensitive, warning_message, categories = detect_sensitive_content(original_text)

    if has_sensitive:
        print(f"⚠️ Sensitive content detected: {categories}")
        # Show warning and wait for user confirmation
        return original_text, "", None, gr.update(value=warning_message, visible=True), gr.update(visible=True)

    # If no sensitive content or user confirmed, proceed with generation
    return continue_generation(original_text, tone, speed, language, enable_tone)

def continue_generation(original_text, tone, speed, language, enable_tone):
    """
    Continue with audiobook generation after confirmation
    """
    # Get language code
    lang_code = LANGUAGES[language]

    # Determine if we should apply tone enhancement
    # Only apply for English and if enabled
    if enable_tone and language == "English":
        # Step 1: Rewrite text with selected tone
        status_update = f"🔄 Applying {tone} tone..."
        print(status_update)

        rewritten_text = generate_rewritten_text(original_text, tone)

        if rewritten_text.startswith("Error"):
            return original_text, rewritten_text, None, gr.update(visible=False), gr.update(visible=False)
    else:
        # Skip tone enhancement for faster processing
        rewritten_text = original_text
        if enable_tone and language != "English":
            rewritten_text = f"[Tone enhancement is only available for English. Using original text for {language} audio generation.]"
        else:
            rewritten_text = "[Tone enhancement disabled. Using original text.]"

    # Step 2: Generate audio in selected language
    status_update = f"🎙️ Generating audio in {language}..."
    print(status_update)

    slow_speech = (speed == "Slow")

    # Use original text for audio if tone enhancement was skipped
    audio_text = original_text if not (enable_tone and language == "English") else rewritten_text
    audio_file = text_to_speech(audio_text, lang_code=lang_code, slow=slow_speech)

    if isinstance(audio_file, str) and audio_file.startswith("Error"):
        return original_text, rewritten_text, None, gr.update(visible=False), gr.update(visible=False)

    return original_text, rewritten_text, audio_file, gr.update(visible=False), gr.update(visible=False)

def proceed_with_generation(input_text, file_input, tone, speed, language, enable_tone):
    """
    User confirmed to proceed despite sensitive content warning
    """
    # Get original text
    if file_input is not None:
        try:
            with open(file_input.name, 'r', encoding='utf-8') as f:
                original_text = f.read()
        except:
            original_text = input_text
    else:
        original_text = input_text

    return continue_generation(original_text, tone, speed, language, enable_tone)

def clear_all():
    """
    Clear all inputs and outputs
    """
    return "", None, "", "", None, gr.update(visible=False), gr.update(visible=False)

# Create Gradio Interface
with gr.Blocks(title="EchoVerse - AI Audiobook Generator", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🎧 EchoVerse - AI Audiobook Generator

    Transform your text into expressive, downloadable audio content with customizable voice and language.
    Powered by **IBM Granite 3.2 2B Instruct** model.

    ### Features:
    - 📝 Paste text or upload .txt files
    - 🎭 Optional tone enhancement (English only, for faster processing)
    - 🌍 Full support for 20+ languages
    - 🎵 Generate natural-sounding narrations
    - ⚠️ Sensitive content detection and warnings
    - ⬇️ Download audio in MP3 format
    - ⚡ Fast processing mode (skip tone enhancement)
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")

            input_text = gr.Textbox(
                label="Paste Your Text",
                placeholder="Enter the text you want to convert to audiobook...",
                lines=8,
                max_lines=15
            )

            file_input = gr.File(
                label="Or Upload .txt File",
                file_types=[".txt"],
                type="filepath"
            )

            # Language Selector
            language_selector = gr.Dropdown(
                choices=list(LANGUAGES.keys()),
                value="English",
                label="🌍 Select Language",
                info="Language for audio narration"
            )

            # Tone Enhancement Toggle
            enable_tone_checkbox = gr.Checkbox(
                label="🎭 Enable Tone Enhancement (English only)",
                value=False,
                info="Rewrite text with AI for better narration. Disable for faster processing."
            )

            with gr.Row():
                tone_selector = gr.Radio(
                    choices=["Neutral", "Suspenseful", "Inspiring"],
                    value="Neutral",
                    label="Select Tone (if enhancement enabled)"
                )

                speed_selector = gr.Radio(
                    choices=["Normal", "Slow"],
                    value="Normal",
                    label="Speech Speed"
                )

            with gr.Row():
                generate_btn = gr.Button("🎙️ Generate Audiobook", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear All", size="lg")

    # Sensitive Content Warning Section
    with gr.Row(visible=False) as warning_row:
        with gr.Column():
            warning_box = gr.Markdown("")

    with gr.Row(visible=False) as warning_buttons_row:
        with gr.Column():
            with gr.Row():
                proceed_btn = gr.Button("⚠️ Proceed Anyway", variant="stop", size="lg")
                cancel_btn = gr.Button("❌ Cancel", size="lg")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Original Text")
            original_output = gr.Textbox(
                label="Your Original Input",
                lines=6,
                interactive=False
            )

        with gr.Column(scale=1):
            gr.Markdown("### Processing Status")
            rewritten_output = gr.Textbox(
                label=f"Tone-Enhanced Version (if enabled)",
                lines=6,
                interactive=False
            )

    with gr.Row():
        audio_output = gr.Audio(
            label="Generated Audiobook",
            type="filepath"
        )

    gr.Markdown("""
    ---
    ### 📚 Usage Instructions:
    1. **Input**: Paste text or upload a .txt file (max 5000 characters)
    2. **Language**: Select the language for audio narration
    3. **Tone Enhancement**: Enable for AI-enhanced text (slower, English only) or disable for faster processing
    4. **Tone**: Select desired tone if enhancement is enabled
    5. **Speed**: Choose narration speed (Normal or Slow)
    6. **Generate**: Click the button to create your audiobook
    7. **Content Warning**: If sensitive content is detected, you'll receive a warning before proceeding
    8. **Download**: Use the download button in the audio player

    ### 💡 Tips:
    - **For fastest results**: Disable tone enhancement and use any language
    - **For enhanced narration**: Enable tone enhancement (works best with English text)
    - Shorter texts (200-1000 words) process faster
    - The Inspiring tone is great for motivational content
    - The Suspenseful tone works well for stories and narratives
    - The Neutral tone is perfect for educational or professional content
    - **Multi-language support**: Your text will be read in the selected language
    - **Content Safety**: The system automatically detects violence, abuse, and strong language

    ### ⚡ Performance Notes:
    - **Tone Enhancement OFF**: ⚡ Super fast - Direct text-to-speech conversion
    - **Tone Enhancement ON**: 🐌 Slower - AI rewrites text for better narration quality

    ### 🌍 Supported Languages:
    English, Spanish, French, German, Italian, Portuguese, Russian, Japanese, Korean, Chinese, Arabic, Hindi, Dutch, Polish, Turkish, Swedish, Indonesian, Thai, Vietnamese, Greek
    """)

    # Event handlers
    generate_btn.click(
        fn=process_audiobook,
        inputs=[input_text, file_input, tone_selector, speed_selector, language_selector, enable_tone_checkbox],
        outputs=[original_output, rewritten_output, audio_output, warning_box, warning_buttons_row]
    ).then(
        fn=lambda w: gr.update(visible=bool(w)),
        inputs=[warning_box],
        outputs=[warning_row]
    )

    proceed_btn.click(
        fn=proceed_with_generation,
        inputs=[input_text, file_input, tone_selector, speed_selector, language_selector, enable_tone_checkbox],
        outputs=[original_output, rewritten_output, audio_output, warning_box, warning_buttons_row]
    ).then(
        fn=lambda: gr.update(visible=False),
        inputs=[],
        outputs=[warning_row]
    )

    cancel_btn.click(
        fn=lambda: (gr.update(visible=False), gr.update(visible=False)),
        inputs=[],
        outputs=[warning_row, warning_buttons_row]
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[input_text, file_input, original_output, rewritten_output, audio_output, warning_row, warning_buttons_row]
    )

# Launch the app
print("\n🚀 Launching EchoVerse with Optimized Performance...")
app.launch(debug=True, share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
Loading IBM Granite model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

...

/tmp/ipykernel_1502/3312829630.py:313: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="EchoVerse - AI Audiobook Generator", theme=gr.themes.Soft()) as app:



🚀 Launching EchoVerse with Optimized Performance...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5465566edeb568b8dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔍 Checking for sensitive content...
🔄 Applying Inspiring tone...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🔄 Applying Neutral tone...
🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🎙️ Generating audio in English...
🔍 Checking for sensitive content...
🎙️ Generating audio in English...
